# 09 — Validacija konačnog integrisanog profila

Završni eksperiment proverava preporučeni profil kao celinu nad svih 40 snimaka čitanog srpskog govora:

**Faster-Whisper Small + INT8 + beam_size=5 + VAD ON + chunk=10 s**

Za svaki snimak računaju se WER, vreme obrade i RTF, a na kraju se računaju i prosečan WER, medijana WER-a i corpus WER.



In [4]:
from pathlib import Path
import time
import re
import unicodedata

import numpy as np
import pandas as pd
from pydub import AudioSegment
from faster_whisper import WhisperModel
from jiwer import wer


In [5]:
# ============================================================
# PUTANJE I KONAČNA KONFIGURACIJA
# ============================================================

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

AUDIO_DIR = PROJECT_ROOT / "audio/40Sentences00-08Notebooks"
REZULTATI_DIR = PROJECT_ROOT / "rezultati" / "09_validacija_konacnog_profila"
REZULTATI_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SIZE = "small"
COMPUTE_TYPE = "int8"
DEVICE = "cpu"

LANGUAGE = "sr"
BEAM_SIZE = 5
VAD_FILTER = True
CHUNK_SECONDS = 10

print("PROJECT_ROOT:", PROJECT_ROOT)
print("AUDIO_DIR:", AUDIO_DIR)
print("REZULTATI_DIR:", REZULTATI_DIR)

print("\nKONAČNI PROFIL:")
print(f"Model:          {MODEL_SIZE}")
print(f"Kvantizacija:   {COMPUTE_TYPE}")
print(f"Uređaj:         {DEVICE}")
print(f"Jezik:          {LANGUAGE}")
print(f"Beam size:      {BEAM_SIZE}")
print(f"VAD:            {VAD_FILTER}")
print(f"Chunk:          {CHUNK_SECONDS} s")


PROJECT_ROOT: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR
AUDIO_DIR: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\audio\40Sentences00-08Notebooks
REZULTATI_DIR: D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\rezultati\09_validacija_konacnog_profila

KONAČNI PROFIL:
Model:          small
Kvantizacija:   int8
Uređaj:         cpu
Jezik:          sr
Beam size:      5
VAD:            True
Chunk:          10 s


## Test skup



In [6]:
# ============================================================
# 20 REPREZENTATIVNIH PRIRODNIH SRPSKIH AUDIO SNIMAKA
#
# kratki:       1-5
# srednji:     11-15
# dugi:        26-30
# veoma dugi:  36-40
# ============================================================

test_set = [
    {
        "broj": 1,
        "audio": AUDIO_DIR / "1govornik1.wav",
        "govornik": "govornik1",
        "grupa": "kratki",
        "ref": "Veštačka inteligencija omogućava računarima da rešavaju složene probleme i analiziraju velike količine podataka."
    },
    {
        "broj": 2,
        "audio": AUDIO_DIR / "2govornik1.wav",
        "govornik": "govornik1",
        "grupa": "kratki",
        "ref": "Automatsko prepoznavanje govora pretvara izgovorene reči u tekst koji računar može dalje da obrađuje."
    },
    {
        "broj": 3,
        "audio": AUDIO_DIR / "3govornik1.wav",
        "govornik": "govornik1",
        "grupa": "kratki",
        "ref": "Danas je veoma prijatan dan, pa smo odlučili da prošetamo kroz centar grada posle ručka."
    },
    {
        "broj": 4,
        "audio": AUDIO_DIR / "4govornik1.wav",
        "govornik": "govornik1",
        "grupa": "kratki",
        "ref": "Novi računari imaju brže procesore, više memorije i znatno bolje performanse nego prethodne generacije."
    },
    {
        "broj": 5,
        "audio": AUDIO_DIR / "5govornik2.wav",
        "govornik": "govornik2",
        "grupa": "kratki",
        "ref": "Mašinsko učenje koristi podatke kako bi model naučio da prepoznaje obrasce i donosi odgovarajuće odluke."
    },
    {
        "broj": 11,
        "audio": AUDIO_DIR / "11govornik1.wav",
        "govornik": "govornik1",
        "grupa": "srednji",
        "ref": "Savremeni sistemi za automatsko prepoznavanje govora koriste duboke neuronske mreže kako bi analizirali zvučni signal i pretvorili ga u tekst. Njihova tačnost zavisi od modela, kvaliteta snimka, jezika i karakteristika govornika."
    },
    {
        "broj": 12,
        "audio": AUDIO_DIR / "12govornik1.wav",
        "govornik": "govornik1",
        "grupa": "srednji",
        "ref": "Juče sam krenuo do grada nešto ranije nego obično. Na ulicama nije bilo mnogo ljudi, pa sam brzo završio sve obaveze. Nakon toga sam seo u obližnji kafić i sačekao prijatelja."
    },
    {
        "broj": 13,
        "audio": AUDIO_DIR / "13govornik1.wav",
        "govornik": "govornik1",
        "grupa": "srednji",
        "ref": "Whisper je model za automatsko prepoznavanje govora koji može da radi sa velikim brojem jezika. Faster-Whisper koristi optimizovanu implementaciju koja omogućava efikasnije izvršavanje i manju potrošnju računarskih resursa."
    },
    {
        "broj": 14,
        "audio": AUDIO_DIR / "14govornik1.wav",
        "govornik": "govornik1",
        "grupa": "srednji",
        "ref": "Kada razgovaramo u tihoj prostoriji, sistem uglavnom može veoma jasno da prepozna izgovorene reči. Međutim, saobraćaj, muzika, razgovor drugih ljudi i različiti izvori šuma mogu značajno da otežaju transkripciju."
    },
    {
        "broj": 15,
        "audio": AUDIO_DIR / "15govornik1.wav",
        "govornik": "govornik1",
        "grupa": "srednji",
        "ref": "Tokom razvoja softverske aplikacije programeri prvo analiziraju zahteve korisnika, zatim projektuju sistem i implementiraju potrebne funkcionalnosti. Nakon toga slede testiranje, ispravljanje grešaka i postavljanje aplikacije u produkciono okruženje."
    },
    {
        "broj": 26,
        "audio": AUDIO_DIR / "26govornik1.wav",
        "govornik": "govornik1",
        "grupa": "dugi",
        "ref": "Automatsko prepoznavanje govora predstavlja oblast veštačke inteligencije koja se bavi pretvaranjem zvučnog signala u tekst. Savremeni sistemi koriste modele dubokog učenja koji mogu da analiziraju kompleksne karakteristike ljudskog govora. Njihove performanse mogu da zavise od jezika, kvaliteta mikrofona, prisustva pozadinskog šuma, brzine govora i karakteristika samog govornika. Zbog toga je prilikom evaluacije važno koristiti različite vrste zvučnih zapisa."
    },
    {
        "broj": 27,
        "audio": AUDIO_DIR / "27govornik1.wav",
        "govornik": "govornik1",
        "grupa": "dugi",
        "ref": "Jednog jutra krenuo sam automobilom prema gradu nešto ranije nego obično. Saobraćaj je u početku bio veoma slab, ali se približavanjem centru broj vozila postepeno povećavao. Dok sam čekao na semaforu, na radiju su govorili o vremenskoj prognozi za narednih nekoliko dana. Najavljeno je sunčano vreme, ali i mogućnost kratkotrajne kiše tokom vikenda. Kada sam stigao, parkirao sam automobil i nastavio pešice."
    },
    {
        "broj": 28,
        "audio": AUDIO_DIR / "28govornik1.wav",
        "govornik": "govornik1",
        "grupa": "dugi",
        "ref": "Whisper postoji u više veličina modela, od veoma malih do znatno većih modela. Manji modeli zahtevaju manje memorije i uglavnom omogućavaju bržu transkripciju, dok veći modeli mogu da pruže veću tačnost u zahtevnijim uslovima. Zbog toga izbor modela predstavlja kompromis između kvaliteta transkripcije i potrebnih računarskih resursa. Taj kompromis je posebno važan kada se sistem izvršava na običnom računaru bez snažne grafičke kartice."
    },
    {
        "broj": 29,
        "audio": AUDIO_DIR / "29govornik2.wav",
        "govornik": "govornik2",
        "grupa": "dugi",
        "ref": "Pozadinski šum predstavlja jedan od najčešćih problema kod sistema za automatsko prepoznavanje govora. U realnom okruženju mikrofon ne snima samo glas osobe već i zvuk automobila, ventilatora, drugih ljudi ili muzike. Ako je govor dovoljno snažan u odnosu na šum, transkripcija može ostati veoma dobra. Međutim, kada se odnos signala i šuma smanjuje, broj pogrešno prepoznatih reči uglavnom počinje da raste."
    },
    {
        "broj": 30,
        "audio": AUDIO_DIR / "30govornik2.wav",
        "govornik": "govornik2",
        "grupa": "dugi",
        "ref": "Kada razvijamo sistem koji treba da radi u realnom vremenu, nije dovoljno posmatrati samo njegovu tačnost. Potrebno je analizirati koliko vremena modelu treba da obradi određenu količinu zvuka. Ako obrada pet sekundi govora traje kraće od pet sekundi, sistem potencijalno može da prati govor bez stalnog povećavanja kašnjenja. Zbog toga su latencija i faktor realnog vremena veoma korisne metrike prilikom evaluacije ovakvih sistema."
    },
    {
        "broj": 36,
        "audio": AUDIO_DIR / "36govornik1.wav",
        "govornik": "govornik1",
        "grupa": "veoma_dugi",
        "ref": "Savremeni sistemi za automatsko prepoznavanje govora imaju veliki broj praktičnih primena. Mogu se koristiti za automatsko pravljenje transkripata sastanaka, generisanje titlova, glasovne asistente, pretragu multimedijalnog sadržaja i poboljšanje pristupačnosti digitalnih sistema. Ipak, kvalitet njihovog rada zavisi od velikog broja faktora. Govornici imaju različite glasove, akcente i brzine govora, dok snimci mogu nastati korišćenjem različitih mikrofona i u različitim prostorijama. Dodatni problem predstavlja pozadinski šum. Sistem koji veoma dobro radi u tihoj prostoriji ne mora da postigne isti rezultat na ulici ili u automobilu. Zbog toga evaluacija sistema ne bi trebalo da bude zasnovana na samo jednom zvučnom zapisu. Potrebno je koristiti više govornika, različite dužine snimaka i različite uslove. Pored tačnosti transkripcije, važno je analizirati vreme izvršavanja, potrošnju memorije i sposobnost sistema da obrađuje zvuk dovoljno brzo za praktičnu primenu."
    },
    {
        "broj": 37,
        "audio": AUDIO_DIR / "37govornik2.wav",
        "govornik": "govornik2",
        "grupa": "veoma_dugi",
        "ref": "Kada govorimo o performansama modela za prepoznavanje govora, postoje dve različite stvari koje želimo da postignemo. Prva je što tačniji transkript, a druga što efikasnije izvršavanje. Veći model može bolje da prepozna komplikovane rečenice ili govor u težim uslovima, ali istovremeno može zahtevati mnogo više memorije i vremena. Manji model može biti nešto manje precizan, ali dovoljno brz da se koristi na običnom laptopu. Kvantizacija predstavlja još jedan način optimizacije jer omogućava korišćenje numeričkog formata manje preciznosti. Time se može smanjiti količina potrebne memorije i ubrzati obrada. Međutim, svaka optimizacija mora da bude eksperimentalno proverena. Nije dovoljno pokazati da je jedna konfiguracija brža. Potrebno je proveriti i da li je njena tačnost ostala prihvatljiva. Upravo zato se koriste metrike kao što su Word Error Rate, latencija i Real-Time Factor, koje omogućavaju da različite konfiguracije modela budu upoređene na objektivan način."
    },
    {
        "broj": 38,
        "audio": AUDIO_DIR / "38govornik2.wav",
        "govornik": "govornik2",
        "grupa": "veoma_dugi",
        "ref": "Prošle nedelje imao sam prilično ispunjen dan. Ujutru sam ustao ranije jer sam morao da završim nekoliko obaveza pre odlaska u grad. Posle doručka proverio sam poruke, pripremio stvari koje su mi bile potrebne i krenuo automobilom. Na putu je bilo više saobraćaja nego što sam očekivao, pa mi je trebalo skoro pola sata da stignem do centra. Prvo sam otišao do prodavnice, zatim do banke, a nakon toga sam se našao sa prijateljem kojeg nisam video nekoliko meseci. Razgovarali smo o poslu, putovanjima i planovima za narednu godinu. Posle ručka vreme se iznenada promenilo i počela je jaka kiša. Sačekali smo nekoliko minuta da se vreme malo smiri, a onda smo krenuli kući. Kada sam se vratio, nastavio sam da radim na računaru. Uveče sam završio preostale obaveze i odlučio da ostatak dana provedem odmarajući se."
    },
    {
        "broj": 39,
        "audio": AUDIO_DIR / "39govornik3.wav",
        "govornik": "govornik3",
        "grupa": "veoma_dugi",
        "ref": "Sistem koji obrađuje govor u realnom vremenu mora da rešava nešto drugačiji problem od sistema koji dobije kompletan audio zapis. Kod klasične transkripcije model može da dobije čitav snimak i zatim ga obradi. Kod realnog vremena zvuk neprestano pristiže, pa ga je potrebno podeliti na manje delove. Ako koristimo veoma kratke segmente, sistem može brzo da prikaže rezultat, ali model ima manje konteksta za pravilno prepoznavanje rečenice. Sa druge strane, ako koristimo duge segmente, model dobija više konteksta, ali korisnik mora duže da čeka na rezultat. Zbog toga izbor veličine segmenta predstavlja kompromis između latencije i kvaliteta transkripcije. Na primer, mogu se uporediti segmenti od dve, pet i deset sekundi. Za svaku konfiguraciju moguće je izmeriti vreme obrade, faktor realnog vremena i tačnost konačnog transkripta. Takav eksperiment omogućava da se utvrdi koja konfiguracija predstavlja najbolji kompromis za praktičan sistem."
    },
    {
        "broj": 40,
        "audio": AUDIO_DIR / "40govornik3.wav",
        "govornik": "govornik3",
        "grupa": "veoma_dugi",
        "ref": "Cilj eksperimentalne analize sistema za automatsko prepoznavanje govora jeste da se utvrdi kako različiti faktori utiču na njegov rad. Možemo menjati veličinu modela, nivo pozadinskog šuma, jezik govornika, parametre dekodiranja, način kvantizacije ili korišćenje algoritma za detekciju govora. Svaki eksperiment treba organizovati tako da možemo jasno da utvrdimo posledice određene promene. Rezultati se zatim mogu posmatrati kroz više metrika. Word Error Rate pokazuje koliko se prepoznati tekst razlikuje od tačnog referentnog transkripta. Latencija pokazuje koliko vremena je potrebno za obradu, dok Real-Time Factor povezuje vreme obrade sa trajanjem samog zvučnog zapisa. Potrošnja memorije je posebno značajna kada model treba da radi na uređajima sa ograničenim resursima. Kombinovanjem ovih rezultata možemo da pronađemo konfiguraciju koja ne mora biti najbolja prema svakoj pojedinačnoj metrici, ali predstavlja dobar kompromis između tačnosti, brzine i potrebnih računarskih resursa."
    }
]

print(f"Broj test audio fajlova: {len(test_set)}")

# Provera postojanja svih fajlova
for item in test_set:
    if not item["audio"].exists():
        raise FileNotFoundError(f"Audio fajl nije pronađen: {item['audio']}")

print("Sva 20 audio fajla su pronađena.")

Broj test audio fajlova: 20
Sva 20 audio fajla su pronađena.


In [7]:
# ============================================================
# NORMALIZACIJA I POMOĆNE FUNKCIJE
# ============================================================

def normalizuj_tekst(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKC", text)

    # Interpunkcija -> razmak
    text = re.sub(r"[^\w\sčćžšđ]", " ", text, flags=re.UNICODE)

    # Višestruki razmaci -> jedan
    text = re.sub(r"\s+", " ", text).strip()

    return text


def govornik_iz_imena(putanja):
    ime = Path(putanja).stem.lower()

    if "govornik1" in ime:
        return "G1"
    elif "govornik2" in ime:
        return "G2"
    elif "govornik3" in ime:
        return "G3"

    return "Nepoznato"


def trajanje_audio(putanja):
    audio = AudioSegment.from_file(str(putanja))
    return len(audio) / 1000.0


In [8]:
# ============================================================
# UČITAVANJE FINALNOG MODELA
# ============================================================

print("Učitavanje finalnog modela...")

model = WhisperModel(
    MODEL_SIZE,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)

print("✓ Model učitan.")


Učitavanje finalnog modela...
✓ Model učitan.


In [9]:
# ============================================================
# TRANSKRIPCIJA U SEGMENTIMA OD 10 SEKUNDI
# ============================================================

def transkribuj_u_chunkovima(
    model,
    audio_path,
    chunk_seconds=10,
    language="sr",
    beam_size=5,
    vad_filter=True
):
    """
    Deli audio zapis na segmente maksimalnog trajanja chunk_seconds,
    transkribuje svaki segment istom konfiguracijom i spaja rezultate.
    """

    audio = AudioSegment.from_file(str(audio_path))
    chunk_ms = int(chunk_seconds * 1000)

    tekstovi = []
    broj_chunkova = 0

    temp_dir = REZULTATI_DIR / "_temp_chunks"
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        for start_ms in range(0, len(audio), chunk_ms):
            end_ms = min(start_ms + chunk_ms, len(audio))
            chunk = audio[start_ms:end_ms]

            broj_chunkova += 1

            temp_path = (
                temp_dir /
                f"{Path(audio_path).stem}_chunk_{broj_chunkova:03d}.wav"
            )

            chunk.export(str(temp_path), format="wav")

            segments, info = model.transcribe(
                str(temp_path),
                language=language,
                task="transcribe",
                beam_size=beam_size,
                temperature=0.0,
                vad_filter=vad_filter,
                condition_on_previous_text=False,
                without_timestamps=False,
                initial_prompt=None
            )

            tekst_chunka = " ".join(
                segment.text.strip()
                for segment in segments
                if segment.text.strip()
            )

            if tekst_chunka:
                tekstovi.append(tekst_chunka)

            if temp_path.exists():
                temp_path.unlink()

    finally:
        try:
            temp_dir.rmdir()
        except OSError:
            pass

    finalni_tekst = " ".join(tekstovi).strip()

    return finalni_tekst, broj_chunkova


## Završno pokretanje na svih 40 snimaka

Merenje vremena obuhvata kompletan postupak obrade jednog snimka kroz sve njegove segmente. Model se učitava samo jednom pre početka benchmarka, tako da vreme učitavanja modela ne ulazi u vreme transkripcije pojedinačnog snimka.


In [10]:
# ============================================================
# GLAVNI EKSPERIMENT — SVIH 40 SNIMAKA
# ============================================================

rezultati = []

print("=" * 100)
print("ZAVRŠNA VALIDACIJA INTEGRISANOG PROFILA")
print("Small INT8 + beam=5 + VAD ON + chunk=10 s")
print("=" * 100)

for i, item in enumerate(test_set, start=1):

    audio_path = Path(item["audio"])
    ref = item["ref"]

    trajanje = trajanje_audio(audio_path)

    print(f"\n[{i:02d}/40] {audio_path.name} ({trajanje:.2f} s)")

    start = time.perf_counter()

    hipoteza, broj_chunkova = transkribuj_u_chunkovima(
        model=model,
        audio_path=audio_path,
        chunk_seconds=CHUNK_SECONDS,
        language=LANGUAGE,
        beam_size=BEAM_SIZE,
        vad_filter=VAD_FILTER
    )

    vreme = time.perf_counter() - start

    ref_norm = normalizuj_tekst(ref)
    hyp_norm = normalizuj_tekst(hipoteza)

    if len(ref_norm) == 0:
        wer_vrednost = np.nan
    else:
        wer_vrednost = wer(ref_norm, hyp_norm) * 100

    rtf = vreme / trajanje if trajanje > 0 else np.nan

    rezultat = {
        "Broj": item.get("broj", i),
        "Fajl": audio_path.name,
        "Govornik": govornik_iz_imena(audio_path),

        "Model": MODEL_SIZE,
        "Kvantizacija": COMPUTE_TYPE,
        "Beam_size": BEAM_SIZE,
        "VAD": VAD_FILTER,
        "Chunk_s": CHUNK_SECONDS,

        "Trajanje_s": trajanje,
        "Broj_chunkova": broj_chunkova,

        "Referenca": ref,
        "Transkript": hipoteza,

        "WER_pct": wer_vrednost,
        "Vreme_s": vreme,
        "RTF": rtf
    }

    rezultati.append(rezultat)

    print(f"  chunkova: {broj_chunkova}")
    print(f"  WER:      {wer_vrednost:.2f}%")
    print(f"  vreme:    {vreme:.2f} s")
    print(f"  RTF:      {rtf:.4f}")

    print(f"  REF: {ref_norm[:150]}")
    print(f"  HYP: {hyp_norm[:150]}")


ZAVRŠNA VALIDACIJA INTEGRISANOG PROFILA
Small INT8 + beam=5 + VAD ON + chunk=10 s

[01/40] 1govornik1.wav (7.61 s)
  chunkova: 1
  WER:      15.38%
  vreme:    4.72 s
  RTF:      0.6211
  REF: veštačka inteligencija omogućava računarima da rešavaju složene probleme i analiziraju velike količine podataka
  HYP: veštočka inteligencija omugućeva računarima da rešavaju složene probleme i analiziraju velike količine podataka

[02/40] 2govornik1.wav (6.78 s)
  chunkova: 1
  WER:      7.14%
  vreme:    3.85 s
  RTF:      0.5684
  REF: automatsko prepoznavanje govora pretvara izgovorene reči u tekst koji računar može dalje da obrađuje
  HYP: automatsko prepoznavanje govora pretvara izgovorene reči u tekst koji računar može dalje da obradjuje

[03/40] 3govornik1.wav (5.83 s)
  chunkova: 1
  WER:      26.67%
  vreme:    3.81 s
  RTF:      0.6535
  REF: danas je veoma prijatan dan pa smo odlučili da prošetamo kroz centar grada posle ručka
  HYP: dana si je veoma prijetan dan pa smo odlučili da pr

In [11]:
# ============================================================
# TABELA REZULTATA I CSV
# ============================================================

df = pd.DataFrame(rezultati)

CSV_PATH = REZULTATI_DIR / "09_validacija_konacnog_profila.csv"

df.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n✓ Rezultati sačuvani:")
print(CSV_PATH)

display(
    df[
        [
            "Broj",
            "Fajl",
            "Govornik",
            "Trajanje_s",
            "Broj_chunkova",
            "WER_pct",
            "Vreme_s",
            "RTF"
        ]
    ].round(4)
)



✓ Rezultati sačuvani:
D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\rezultati\09_validacija_konacnog_profila\09_validacija_konacnog_profila.csv


,Broj,Fajl,Govornik,Trajanje_s,Broj_chunkova,WER_pct,Vreme_s,RTF
0,1,1govornik1.wav,G1,7.605,1,15.3846,4.7237,0.6211
1,2,2govornik1.wav,G1,6.780,1,7.1429,3.8537,0.5684
2,3,3govornik1.wav,G1,5.828,1,26.6667,3.8088,0.6535
3,4,4govornik1.wav,G1,7.291,1,21.4286,3.7399,0.5129
4,5,5govornik2.wav,G2,9.334,1,13.3333,4.0250,0.4312
5,11,11govornik1.wav,G1,16.463,2,22.5806,8.1254,0.4936
6,12,12govornik1.wav,G1,12.725,2,22.5806,8.5964,0.6756
7,13,13govornik1.wav,G1,15.081,2,31.0345,7.7568,0.5143
8,14,14govornik1.wav,G1,15.604,2,34.4828,7.9110,0.5070
9,15,15govornik1.wav,G1,17.427,2,32.1429,7.8032,0.4478


In [12]:
# ============================================================
# CORPUS WER
# ============================================================

sve_reference = " ".join(
    normalizuj_tekst(x)
    for x in df["Referenca"].tolist()
)

sve_hipoteze = " ".join(
    normalizuj_tekst(x)
    for x in df["Transkript"].tolist()
)

corpus_wer = wer(
    sve_reference,
    sve_hipoteze
) * 100

print(f"Corpus WER: {corpus_wer:.2f}%")


Corpus WER: 25.27%


In [13]:
# ============================================================
# KONAČNI REZIME ZA MASTER RAD
# ============================================================

rezime = pd.DataFrame([{
    "Konfiguracija": "Small INT8 + beam 5 + VAD + chunk 10 s",

    "Broj_snimaka": len(df),

    "Prosecan_WER_pct": df["WER_pct"].mean(),
    "Medijana_WER_pct": df["WER_pct"].median(),
    "Corpus_WER_pct": corpus_wer,

    "Prosecno_vreme_s": df["Vreme_s"].mean(),
    "Medijana_vremena_s": df["Vreme_s"].median(),
    "Ukupno_vreme_s": df["Vreme_s"].sum(),

    "Prosecan_RTF": df["RTF"].mean(),
    "Medijana_RTF": df["RTF"].median(),
    "Maksimalni_RTF": df["RTF"].max(),

    "Ukupno_chunkova": df["Broj_chunkova"].sum()
}])

display(rezime.round(4))

REZIME_PATH = (
    REZULTATI_DIR /
    "09_validacija_konacnog_profila_rezime.csv"
)

rezime.to_csv(
    REZIME_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSačuvan rezime:")
print(REZIME_PATH)


,Konfiguracija,Broj_snimaka,Prosecan_WER_pct,Medijana_WER_pct,Corpus_WER_pct,Prosecno_vreme_s,Medijana_vremena_s,Ukupno_vreme_s,Prosecan_RTF,Medijana_RTF,Maksimalni_RTF,Ukupno_chunkova
0,Small INT8 + beam 5 + VAD + chunk 10 s,20,23.436,22.5806,25.2693,14.728,10.6172,294.5592,0.4994,0.4945,0.6756,72



Sačuvan rezime:
D:\0000000000MasterRad\aleksandar_jovanović_Faster-Whisper&CTranslate2_2026\Projekat2ASR\rezultati\09_validacija_konacnog_profila\09_validacija_konacnog_profila_rezime.csv


## Pregled najtežih slučajeva

Pored agregatnih metrika prikazuje se deset snimaka sa najvećim WER-om, zajedno sa referencom i dobijenim transkriptom. Ovaj pregled može se koristiti i za kvalitativnu proveru ekstremnih grešaka i mogućih halucinacija.


In [14]:
# ============================================================
# 10 SNIMAKA SA NAJVEĆIM WER-om
# ============================================================

print("10 SNIMAKA SA NAJVEĆIM WER-om:")

najgori = (
    df.sort_values("WER_pct", ascending=False)
      .head(10)
)

display(
    najgori[
        [
            "Broj",
            "Fajl",
            "Govornik",
            "Trajanje_s",
            "Broj_chunkova",
            "WER_pct",
            "RTF"
        ]
    ].round(4)
)

for _, row in najgori.iterrows():

    print("\n" + "=" * 100)

    print(
        f"{row['Fajl']} | "
        f"WER={row['WER_pct']:.2f}% | "
        f"RTF={row['RTF']:.4f}"
    )

    print("\nREF:")
    print(row["Referenca"])

    print("\nHYP:")
    print(row["Transkript"])


10 SNIMAKA SA NAJVEĆIM WER-om:


,Broj,Fajl,Govornik,Trajanje_s,Broj_chunkova,WER_pct,RTF
18,39,39govornik3.wav,G3,62.613,7,38.8489,0.4954
10,26,26govornik1.wav,G1,31.092,4,34.4828,0.5341
8,14,14govornik1.wav,G1,15.604,2,34.4828,0.5070
15,36,36govornik1.wav,G1,69.010,7,33.0769,0.4310
9,15,15govornik1.wav,G1,17.427,2,32.1429,0.4478
7,13,13govornik1.wav,G1,15.081,2,31.0345,0.5143
12,28,28govornik1.wav,G1,30.697,4,26.9841,0.5081
2,3,3govornik1.wav,G1,5.828,1,26.6667,0.6535
19,40,40govornik3.wav,G3,79.772,8,25.3731,0.4284
6,12,12govornik1.wav,G1,12.725,2,22.5806,0.6756



39govornik3.wav | WER=38.85% | RTF=0.4954

REF:
Sistem koji obrađuje govor u realnom vremenu mora da rešava nešto drugačiji problem od sistema koji dobije kompletan audio zapis. Kod klasične transkripcije model može da dobije čitav snimak i zatim ga obradi. Kod realnog vremena zvuk neprestano pristiže, pa ga je potrebno podeliti na manje delove. Ako koristimo veoma kratke segmente, sistem može brzo da prikaže rezultat, ali model ima manje konteksta za pravilno prepoznavanje rečenice. Sa druge strane, ako koristimo duge segmente, model dobija više konteksta, ali korisnik mora duže da čeka na rezultat. Zbog toga izbor veličine segmenta predstavlja kompromis između latencije i kvaliteta transkripcije. Na primer, mogu se uporediti segmenti od dve, pet i deset sekundi. Za svaku konfiguraciju moguće je izmeriti vreme obrade, faktor realnog vremena i tačnost konačnog transkripta. Takav eksperiment omogućava da se utvrdi koja konfiguracija predstavlja najbolji kompromis za praktičan sistem.



## Tumačenje

Numerički zaključak za master rad unosi se tek nakon izvršavanja svih ćelija. Posebno treba navesti prosečan WER, corpus WER, prosečno vreme i prosečan RTF konačnog integrisanog profila.

Ovaj eksperiment predstavlja završnu validaciju profila kao celine i dopunjuje prethodne odvojene eksperimente za izbor modela, kvantizacije, Beam Search parametra, VAD-a i veličine segmenta.
